# Notebook 3: Evaluating Generated Materials — MLIPs

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Understand why **evaluation** is critical after generation
- Use a **Machine Learning Interatomic Potential** ([CHGNet](https://github.com/CederGroupHub/chgnet)) to evaluate structures
- Predict energies, forces, and stresses in seconds (vs. hours with DFT)
- **Relax a perturbed crystal** and visualize the energy trajectory + atomic motion
- Compute **phonon DOS** to check dynamic stability
- Estimate **energy above hull** ($E_\text{hull}$) for thermodynamic stability
- Compare CHGNet predictions against DFT reference data

> **Prerequisites:** Run `00_setup.ipynb` first (CHGNet was installed there).

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 3.1 The evaluation problem

Generating crystal structures is only half the job. After generation, we need to ask:

1. **Is this structure physically reasonable?** (sensible bond lengths, coordination)
2. **Is it thermodynamically stable?** (low energy, near the convex hull)
3. **Does it have interesting properties?** (magnetic, electronic, mechanical)

### The traditional approach: DFT
Density Functional Theory (DFT) gives accurate energies and forces, but it's
**expensive** — each structure takes minutes to hours on a supercomputer.
If you generated 10,000 candidates, DFT relaxation could take weeks.

### The modern approach: MLIPs
Machine Learning Interatomic Potentials are trained on DFT data and can predict
energies/forces/stresses in **seconds** — 1000x faster than DFT.
They enable rapid screening of large candidate sets.

---
## 3.2 CHGNet: a universal MLIP

[CHGNet](https://github.com/CederGroupHub/chgnet) (Crystal Hamiltonian Graph Neural Network)
is a universal potential trained on the Materials Project database.
It can predict energies, forces, stresses, and magnetic moments for any inorganic material.

Let's load the pretrained model.

In [ ]:
from chgnet.model.model import CHGNet

chgnet = CHGNet.load()
print(f'CHGNet loaded: {sum(p.numel() for p in chgnet.parameters()):,} parameters')

---
## 3.3 Predicting properties

Let's predict the energy of a known structure from the training data.

In [ ]:
import os
import pandas as pd
from pymatgen.core import Structure

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

# Pick a structure
row = df.iloc[0]
struct = Structure.from_str(row['cif'], fmt='cif')

print(f'Structure: {row["pretty_formula"]} ({row["material_id"]})')
print(f'Atoms: {struct.num_sites}')
print(f'DFT formation energy: {row["formation_energy_per_atom"]:.4f} eV/atom')

In [ ]:
# Predict with CHGNet
prediction = chgnet.predict_structure(struct)

print(f'CHGNet predictions for {row["pretty_formula"]}:')
print(f'  Energy:  {prediction["e"]:>10.4f} eV/atom')
print(f'  Forces:  max |F| = {abs(prediction["f"]).max():.4f} eV/Å')
print(f'  Stress:  {prediction["s"].diagonal().tolist()}')
if 'm' in prediction:
    print(f'  Magmom:  {prediction["m"].tolist()}')

---
## 3.4 Structure relaxation from a perturbed state

To demonstrate relaxation clearly, we start from a **deliberately perturbed**
structure — we take a known stable crystal and displace atoms randomly by ~0.3 Å.
This simulates what a generated (unrelaxed) structure might look like.

Then we relax with CHGNet and watch:
1. The **energy decrease** step by step toward equilibrium
2. The **atoms settle** back to their stable positions

In [ ]:
import numpy as np
from chgnet.model.dynamics import StructOptimizer

# --- Create a perturbed (unstable) structure ---
np.random.seed(42)
perturbed = struct.copy()

# Add random displacements of ~0.3 Å to each atom
noise_amplitude = 0.3  # Å
cart_coords = perturbed.cart_coords.copy()
noise = noise_amplitude * np.random.randn(*cart_coords.shape)
for i, site in enumerate(perturbed):
    perturbed.translate_sites(i, noise[i], frac_coords=False)

# Check: energy of perturbed structure is HIGHER than the original
pred_orig = chgnet.predict_structure(struct)
pred_pert = chgnet.predict_structure(perturbed)
print(f'Original (equilibrium):  E = {pred_orig["e"]:.4f} eV/atom,  max|F| = {abs(pred_orig["f"]).max():.4f} eV/Å')
print(f'Perturbed (noisy):       E = {pred_pert["e"]:.4f} eV/atom,  max|F| = {abs(pred_pert["f"]).max():.2f} eV/Å')
print(f'Energy increase from perturbation: {pred_pert["e"] - pred_orig["e"]:.4f} eV/atom')
print(f'\nRelaxing perturbed structure with CHGNet...')

# --- Relax the perturbed structure ---
optimizer = StructOptimizer()
result = optimizer.relax(perturbed, verbose=True)

relaxed = result['final_structure']
e_final = result['trajectory'].energies[-1] / relaxed.num_sites
print(f'\nRelaxation complete:')
print(f'  Perturbed energy: {pred_pert["e"]:.4f} eV/atom')
print(f'  Relaxed energy:   {e_final:.4f} eV/atom')
print(f'  DFT reference:    {row["formation_energy_per_atom"]:.4f} eV/atom (formation energy)')
print(f'  Energy drop:      {pred_pert["e"] - e_final:.4f} eV/atom')

In [ ]:
# Compare: how close did relaxation get to the original equilibrium structure?
print('=== Comparison: perturbed → relaxed vs. original equilibrium ===\n')

print('Lattice parameter changes (relaxed vs. original):')
for param in ['a', 'b', 'c', 'alpha', 'beta', 'gamma']:
    orig = getattr(struct.lattice, param)
    relax = getattr(relaxed.lattice, param)
    unit = 'Å' if param in ['a', 'b', 'c'] else '°'
    print(f'  {param:>5s}: orig={orig:8.3f}  relaxed={relax:8.3f} {unit}  (Δ = {relax - orig:+.4f})')

# Displacement: perturbed → relaxed (how far did atoms move during relaxation?)
disp_relax = np.linalg.norm(relaxed.cart_coords - perturbed.cart_coords, axis=1)
# Displacement: relaxed → original (how close to equilibrium did we get?)
disp_equil = np.linalg.norm(relaxed.cart_coords - struct.cart_coords, axis=1)
# Original perturbation magnitude
disp_noise = np.linalg.norm(perturbed.cart_coords - struct.cart_coords, axis=1)

species_list = [str(s.specie) for s in struct]
print(f'\nPer-atom displacements:')
print(f'  {"Atom":<6} {"Perturbation":<16} {"Relaxation move":<18} {"Residual from eq.":<18}')
for i in range(struct.num_sites):
    print(f'  {species_list[i]:<6} {disp_noise[i]:>10.4f} Å      {disp_relax[i]:>10.4f} Å        {disp_equil[i]:>10.4f} Å')

print(f'\nMean perturbation:      {disp_noise.mean():.4f} Å')
print(f'Mean residual from eq.: {disp_equil.mean():.4f} Å')
print(f'Recovery ratio:         {(1 - disp_equil.mean() / disp_noise.mean()) * 100:.1f}%')

### Relaxation trajectory

Let's visualize how the energy decreased during relaxation:

In [ ]:
import matplotlib.pyplot as plt

# Plot energy trajectory during relaxation
energies_traj = result['trajectory'].energies
n_atoms_relax = relaxed.num_sites
e_per_atom = [e / n_atoms_relax for e in energies_traj]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Energy trajectory
ax = axes[0]
steps = range(len(e_per_atom))
ax.plot(steps, e_per_atom, 'o-', color='steelblue', markersize=5, linewidth=1.5,
        label='CHGNet relaxation')
ax.axhline(pred_orig['e'], color='green', linestyle='--', alpha=0.7,
           label=f'Original equilibrium ({pred_orig["e"]:.4f} eV/atom)')
ax.axhline(pred_pert['e'], color='red', linestyle=':', alpha=0.7,
           label=f'Perturbed start ({pred_pert["e"]:.4f} eV/atom)')
ax.set_xlabel('Optimization step', fontsize=12)
ax.set_ylabel('Energy (eV/atom)', fontsize=12)
ax.set_title('Relaxation trajectory: energy minimization', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Max force trajectory (if available)
ax2 = axes[1]
try:
    forces_traj = result['trajectory'].forces
    max_forces = [np.max(np.linalg.norm(f, axis=1)) for f in forces_traj]
    ax2.semilogy(range(len(max_forces)), max_forces, 'o-', color='coral',
                 markersize=5, linewidth=1.5)
    ax2.axhline(0.05, color='gray', linestyle='--', alpha=0.5,
                label='Convergence threshold (0.05 eV/Å)')
    ax2.set_xlabel('Optimization step', fontsize=12)
    ax2.set_ylabel('Max |Force| (eV/Å)', fontsize=12)
    ax2.set_title('Force convergence', fontsize=13)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
except Exception:
    ax2.text(0.5, 0.5, 'Force trajectory not available', ha='center', va='center',
             transform=ax2.transAxes, fontsize=12)

plt.tight_layout()
plt.show()

print(f'Converged in {len(energies_traj)} steps')
print(f'Energy drop: {e_per_atom[0] - e_per_atom[-1]:.4f} eV/atom')

### Structure animation during relaxation

The animation below shows three things simultaneously:
1. **Colored spheres** — current atomic positions at each optimization step
2. **Red arrows** — forces acting on each atom (direction the optimizer pushes them)
3. **Small green dots** — equilibrium positions (from the original unperturbed structure)

Watch how the atoms (spheres) migrate toward the equilibrium positions (green dots)
while the force arrows shrink to zero.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np

# Extract trajectory frames
traj = result['trajectory']
n_frames = len(traj.energies)

# Select key frames (evenly spaced, max 20 frames for smooth animation)
n_show = min(n_frames, 20)
frame_indices = np.linspace(0, n_frames - 1, n_show, dtype=int)

# Equilibrium positions (original unperturbed structure)
equil_pos = struct.cart_coords

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')

# Get coordinate range for consistent axis limits
all_positions = np.array([traj.atom_positions[i] for i in frame_indices])
pad = 0.5
x_lim = (all_positions[:, :, 0].min() - pad, all_positions[:, :, 0].max() + pad)
y_lim = (all_positions[:, :, 1].min() - pad, all_positions[:, :, 1].max() + pad)
z_lim = (all_positions[:, :, 2].min() - pad, all_positions[:, :, 2].max() + pad)

# Force scale: normalize arrows for visibility
has_forces = hasattr(traj, 'forces') and traj.forces is not None and len(traj.forces) > 0
if has_forces:
    all_forces = np.array([traj.forces[i] for i in frame_indices])
    max_force_global = np.max(np.linalg.norm(all_forces, axis=2))
    arrow_scale = 1.0 / max(max_force_global, 1e-6)  # normalize to ~1 Å length

species_list = [str(s.specie) for s in struct]
unique_el = sorted(set(species_list))
cmap_colors = plt.cm.tab10(np.linspace(0, 0.9, max(len(unique_el), 1)))
color_map = {el: cmap_colors[i] for i, el in enumerate(unique_el)}

def update(frame_num):
    ax.clear()
    idx = frame_indices[frame_num]
    positions = traj.atom_positions[idx]
    energy = traj.energies[idx] / struct.num_sites

    # Draw equilibrium positions (small green dots)
    ax.scatter(equil_pos[:, 0], equil_pos[:, 1], equil_pos[:, 2],
               s=30, c='limegreen', alpha=0.6, marker='o',
               edgecolors='green', linewidths=0.5, label='Equilibrium')

    # Draw current atom positions
    for el in unique_el:
        mask = [s == el for s in species_list]
        coords = positions[mask]
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                   s=200, c=[color_map[el]], label=el,
                   alpha=0.85, edgecolors='k', linewidths=0.5)

    # Draw force arrows
    if has_forces and idx < len(traj.forces):
        forces = traj.forces[idx]
        for i in range(len(positions)):
            f = forces[i]
            f_mag = np.linalg.norm(f)
            if f_mag > 0.01:  # skip tiny forces
                ax.quiver(positions[i, 0], positions[i, 1], positions[i, 2],
                          f[0] * arrow_scale, f[1] * arrow_scale, f[2] * arrow_scale,
                          color='red', alpha=0.7, linewidth=1.5,
                          arrow_length_ratio=0.3)

    ax.set_xlim(*x_lim); ax.set_ylim(*y_lim); ax.set_zlim(*z_lim)
    ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_zlabel('z (Å)')

    # Build legend without duplicates
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=7)

    max_f = np.max(np.linalg.norm(traj.forces[idx], axis=1)) if has_forces and idx < len(traj.forces) else 0
    ax.set_title(f'Step {idx}/{n_frames-1}  |  E = {energy:.4f} eV/atom  |  max|F| = {max_f:.3f} eV/Å',
                 fontsize=10)

anim = FuncAnimation(fig, update, frames=n_show, interval=400, repeat=True)
plt.close(fig)

try:
    display(HTML(anim.to_jshtml()))
except Exception:
    # Fallback: show first, middle, and last frames as static plots
    print('Animation display not supported — showing key frames instead.')
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={'projection': '3d'})
    for ax2, label, fi in zip(axes2, ['Initial (perturbed)', 'Mid relaxation', 'Final (relaxed)'],
                               [0, n_show // 2, n_show - 1]):
        idx = frame_indices[fi]
        positions = traj.atom_positions[idx]
        energy = traj.energies[idx] / struct.num_sites
        # Equilibrium
        ax2.scatter(equil_pos[:, 0], equil_pos[:, 1], equil_pos[:, 2],
                    s=30, c='limegreen', alpha=0.6, edgecolors='green', linewidths=0.5)
        # Current positions
        for el in unique_el:
            mask = [s == el for s in species_list]
            coords = positions[mask]
            ax2.scatter(coords[:, 0], coords[:, 1], coords[:, 2],
                       s=200, c=[color_map[el]], label=el,
                       alpha=0.85, edgecolors='k', linewidths=0.5)
        # Forces
        if has_forces and idx < len(traj.forces):
            forces = traj.forces[idx]
            for i in range(len(positions)):
                f = forces[i]
                if np.linalg.norm(f) > 0.01:
                    ax2.quiver(positions[i, 0], positions[i, 1], positions[i, 2],
                              f[0]*arrow_scale, f[1]*arrow_scale, f[2]*arrow_scale,
                              color='red', alpha=0.7, linewidth=1.5, arrow_length_ratio=0.3)
        ax2.set_xlim(*x_lim); ax2.set_ylim(*y_lim); ax2.set_zlim(*z_lim)
        ax2.set_xlabel('x (Å)'); ax2.set_ylabel('y (Å)'); ax2.set_zlabel('z (Å)')
        ax2.set_title(f'{label}\n(step {idx}, E={energy:.4f})', fontsize=10)
    plt.tight_layout()
    plt.show()

print('Green dots = equilibrium positions | Red arrows = forces | Spheres = current positions')

---
## 3.5 Batch evaluation

One of the key advantages of MLIPs is speed. Let's evaluate many structures
and compare CHGNet formation energies against DFT values.

In [ ]:
import matplotlib.pyplot as plt
import time
import numpy as np

# Evaluate the first 20 structures in the test set
n_eval = min(20, len(df))
energies_chgnet = []
energies_dft = []
formulas = []

# Compute CHGNet elemental reference energies for formation energy calculation
# (energy of each element in its standard bulk form)
from pymatgen.core import Structure, Lattice, Element

def get_elemental_ref_energies(chgnet_model):
    """Get CHGNet total energy per atom for common elemental reference structures."""
    refs = {}
    # Use simple structures as references
    # In practice, the MP elemental references are more accurate,
    # but this gives a reasonable approximation for comparison
    for el_symbol in ['Li', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ti', 'V', 'Cr',
                      'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se',
                      'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag',
                      'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 'Ba', 'La', 'Hf', 'Ta',
                      'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Pb', 'Bi',
                      'O', 'S', 'N', 'F', 'Cl', 'Br', 'I', 'P', 'C', 'B', 'H']:
        try:
            el = Element(el_symbol)
            # BCC structure as rough reference
            lat = Lattice.cubic(3.0)
            s = Structure(lat, [el_symbol, el_symbol], [[0,0,0], [0.5,0.5,0.5]])
            pred = chgnet_model.predict_structure(s)
            refs[el_symbol] = float(pred['e'])
        except:
            pass
    return refs

print('Computing elemental reference energies...')
elem_refs = get_elemental_ref_energies(chgnet)
print(f'  References computed for {len(elem_refs)} elements')

start = time.time()
for i in range(n_eval):
    row = df.iloc[i]
    try:
        s = Structure.from_str(row['cif'], fmt='cif')
        pred = chgnet.predict_structure(s)

        # Compute formation energy: E_total - sum(x_i * E_ref_i)
        total_e = float(pred['e'])  # eV/atom
        comp = s.composition.fractional_composition
        ref_sum = sum(comp[el] * elem_refs.get(str(el), 0) for el in comp)
        form_e = total_e - ref_sum

        energies_chgnet.append(form_e)
        energies_dft.append(row['formation_energy_per_atom'])
        formulas.append(row['pretty_formula'])
    except Exception as e:
        print(f'  Skipping {row["pretty_formula"]}: {e}')

elapsed = time.time() - start
print(f'Evaluated {len(energies_chgnet)} structures in {elapsed:.1f}s')
print(f'({elapsed/len(energies_chgnet)*1000:.0f} ms per structure)')

### Speed comparison: DFT vs. MLIP

One of the key advantages of MLIPs — they're orders of magnitude faster than DFT:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

methods = ['DFT\n(VASP/QE)', 'CHGNet\n(MLIP)']
times = [3600, elapsed / len(energies_chgnet)]  # seconds per structure
colors = ['#E69F00', 'steelblue']

bars = ax.barh(methods, times, color=colors, edgecolor='white', height=0.5)
ax.set_xscale('log')
ax.set_xlabel('Time per structure (seconds)', fontsize=11)
ax.set_title('DFT vs. MLIP evaluation speed', fontsize=13)

# Annotate bars
for bar, t in zip(bars, times):
    if t >= 60:
        label = f'{t/60:.0f} min'
    else:
        label = f'{t:.2f} s'
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0.001, 50000)
plt.tight_layout()
plt.show()

speedup = 3600 / (elapsed / len(energies_chgnet))
print(f'CHGNet is ~{speedup:.0f}x faster than DFT for energy evaluation')

In [ ]:
if not energies_dft:
    print('No data — run the batch evaluation cell above first.')
else:
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(energies_dft, energies_chgnet, s=50, c='steelblue',
               edgecolors='white', linewidths=0.5)

    # Diagonal line
    all_vals = energies_dft + energies_chgnet
    lims = [min(all_vals) - 0.2, max(all_vals) + 0.2]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect agreement')

    # Compute MAE
    mae = np.mean(np.abs(np.array(energies_dft) - np.array(energies_chgnet)))

    ax.set_xlabel('DFT formation energy (eV/atom)', fontsize=12)
    ax.set_ylabel('CHGNet formation energy (eV/atom)', fontsize=12)
    ax.set_title(f'CHGNet vs. DFT — Formation Energy\nMAE = {mae:.3f} eV/atom', fontsize=13)
    ax.legend()
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

    print(f'Note: CHGNet elemental references are approximate (BCC structures).')
    print(f'For production use, use MP-computed elemental reference energies.')

---
## 3.6 Phonon calculation with CHGNet

Phonons (lattice vibrations) are a fundamental test of structural stability:
- **No imaginary frequencies** → structure is at a true energy minimum (dynamically stable)
- **Imaginary frequencies** → structure is a saddle point (dynamically unstable)

We use the [ASE Phonons](https://wiki.fysik.dtu.dk/ase/ase/phonons.html) module
with CHGNet as the force calculator. This computes the **dynamical matrix** via
finite displacements and extracts the phonon density of states (DOS).

> **Note:** For production phonon calculations, use [Phonopy](https://phonopy.github.io/phonopy/)
> with larger supercells. The ASE approach here is a quick tutorial demonstration.

---
## 3.8 Connection to SCIGEN's screening pipeline

SCIGEN includes its own GNN-based screening models (in `gnn_eval/`) that evaluate:
- **Stability** — Is the structure near the convex hull?
- **Magnetism** — Does the structure have interesting magnetic properties?

The full SCIGEN pipeline:
1. **Generate** — Create candidates with structural constraints (Notebook 05)
2. **Pre-screen** — Filter by composition validity, bond distances (`script/eval_screen.py`)
3. **MLIP screen** — Evaluate with GNN models or CHGNet
4. **DFT validation** — Final verification of the best candidates

In the published work, this pipeline narrowed 10 million generated structures
down to 24,743 DFT-validated candidates.

---
## 3.7 Thermodynamic stability: energy above the convex hull

The **convex hull** of formation energies defines the set of thermodynamically stable
phases. A structure's **energy above hull** ($E_\text{hull}$) measures how far it is
from the nearest stable decomposition:

- $E_\text{hull} = 0$ → on the hull (thermodynamically stable)
- $E_\text{hull} < 0.025$ eV/atom → likely synthesizable (metastable)
- $E_\text{hull} > 0.1$ eV/atom → likely unstable (will decompose)

We use pymatgen's `PhaseDiagram` with the test set entries as reference phases.
This is an approximation — for production use, load the full Materials Project convex hull.

> **Note:** Computing E_hull against the full MP database requires an API key
> (`mp-api`). Here we build a local convex hull from the test set itself as a
> demonstration of the concept.

In [ ]:
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
from pymatgen.core import Composition

# Build a phase diagram from the test set (using DFT formation energies as ground truth)
# In production, you'd use the full Materials Project database
n_hull = min(200, len(df))
pd_entries = []

for i in range(n_hull):
    r = df.iloc[i]
    try:
        comp = Composition(r['pretty_formula'])
        # PDEntry wants total energy per atom (not formation energy)
        # formation_energy_per_atom is relative to elemental references
        pd_entries.append(PDEntry(comp, r['formation_energy_per_atom'],
                                  name=r.get('material_id', f'mp-{i}')))
    except Exception:
        pass

print(f'Built phase diagram from {len(pd_entries)} entries')

try:
    phase_diag = PhaseDiagram(pd_entries)
    print(f'Stable phases: {len(phase_diag.stable_entries)}')
    print(f'Elements in hull: {[str(e) for e in phase_diag.elements]}')

    # Compute E_hull for the first few structures
    print(f'\n{"#":>3} {"Formula":<20} {"E_form (DFT)":<14} {"E_hull":<10} {"Status"}')
    print('-' * 65)

    e_hulls_dft = []
    for i in range(min(20, len(pd_entries))):
        entry = pd_entries[i]
        try:
            e_hull = phase_diag.get_e_above_hull(entry)
            e_hulls_dft.append(e_hull)
            status = 'STABLE' if e_hull < 1e-4 else ('metastable' if e_hull < 0.025 else 'unstable')
            print(f'{i:3d} {entry.name:<20} {entry.energy_per_atom:>10.4f}    {e_hull:>8.4f}  {status}')
        except Exception:
            e_hulls_dft.append(None)

    # Plot E_hull distribution
    valid_hulls = [h for h in e_hulls_dft if h is not None]
    if valid_hulls:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(valid_hulls, bins=20, color='steelblue', edgecolor='white')
        ax.axvline(0, color='green', linewidth=2, label='Convex hull (stable)')
        ax.axvline(0.025, color='orange', linestyle='--', label='Metastability threshold')
        ax.axvline(0.1, color='red', linestyle='--', label='Instability threshold')
        ax.set_xlabel('Energy above hull (eV/atom)', fontsize=12)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title('Thermodynamic stability distribution', fontsize=13)
        ax.legend(fontsize=9)
        ax.set_xlim(-0.01, max(valid_hulls) + 0.02)
        plt.tight_layout()
        plt.show()

except Exception as e:
    print(f'Phase diagram construction failed: {e}')
    print('This can happen if the test set lacks elemental reference entries.')
    print('For full E_hull analysis, use mp-api to load the Materials Project hull.')

---
## Key takeaways

1. **Generation without evaluation is useless.** You need fast, reliable ways to screen candidates.
2. **MLIPs (like CHGNet) are 1000x faster than DFT** and accurate enough for initial screening.
3. **Relaxation from perturbed structures** demonstrates that CHGNet can recover equilibrium positions — critical for evaluating generated (unrelaxed) structures.
4. **Phonon calculations** verify dynamic stability — no imaginary frequencies means the structure is at a true energy minimum.
5. **Energy above hull ($E_\text{hull}$)** quantifies thermodynamic stability against decomposition.
6. **SCIGEN's pipeline** combines generation → pre-screening → MLIP evaluation → DFT validation.

In the next notebook, we'll generate crystal structures with SCIGEN!

---
## References

- **CHGNet:** Deng et al., "CHGNet as a pretrained universal neural network potential for charge-informed atomistic modelling," *Nature Machine Intelligence* 5, 1031–1041 (2023). [GitHub](https://github.com/CederGroupHub/chgnet)
- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y)
- **Materials Project:** Jain et al., "Commentary: The Materials Project: A materials genome approach to accelerating materials innovation," *APL Materials* 1, 011002 (2013). [materialsproject.org](https://materialsproject.org)
- **ASE:** Larsen et al., "The Atomic Simulation Environment—a Python library for working with atoms," *J. Phys.: Condens. Matter* 29, 273002 (2017). [wiki.fysik.dtu.dk/ase](https://wiki.fysik.dtu.dk/ase/)
- **Phonopy:** Togo & Tanaka, "First principles phonon calculations in materials science," *Scr. Mater.* 108, 1–5 (2015). [phonopy.github.io](https://phonopy.github.io/phonopy/)
- **Convex hull / thermodynamic stability:** Sun et al., "Thermodynamic Stability Trend of Cubic Perovskites," *J. Am. Chem. Soc.* 139, 14905 (2017).

---
## What's next?

Proceed to **Notebook 04: SCIGEN Generation** — the capstone demo where we generate crystal structures with targeted lattice constraints.

In [ ]:
# Your code here


---
## Key takeaways

1. **Generation without evaluation is useless.** You need fast, reliable ways to screen candidates.
2. **MLIPs (like CHGNet) are 1000x faster than DFT** and accurate enough for initial screening.
3. **Relaxation** lets you check if a generated structure is near an energy minimum.
4. **SCIGEN's pipeline** combines generation → pre-screening → MLIP evaluation → DFT validation.

In the next notebook, we'll generate crystal structures with SCIGEN!

---
## What's next?

Proceed to **Notebook 04: SCIGEN Generation** — the capstone demo where we generate crystal structures with targeted lattice constraints.